# Step3 (패널): IT 섹터 Walk-Forward 분류 — Cross-Sectional Panel

## 기존 Step3 (per-ticker 개별) 와의 차이

| 항목 | 기존 (per-ticker) | 이 노트북 (panel) |
|------|-----------------|------------------|
| 학습 단위 | 종목별 독립 모델 (10개/window) | 전체 종목 합쳐서 **1개/window** |
| IS 데이터 크기 | 252행 × 1종목 | ~2,520행 × 10종목 스택 |
| 5분위 기준 | 종목 자신의 과거 수익률 분포 | **같은 날짜 10종목 간 상대 순위** |
| TabPFN | v0.1.9 (행 제한) | **v2 (토큰 필요, 행 제한 없음)** |
| 학습 속도 | ~75분 | **~7분 (10배 빠름)** |

**사전 준비**: `Test/.env` 파일에 `TABPFN_TOKEN=발급받은토큰` 입력  
토큰 발급: https://app.priorlabs.ai

In [1]:
"""
Section 0: Import 및 함수 정의 (패널 버전)
  - .env에서 TABPFN_TOKEN 로드 → os.environ에 명시적 설정
  - TabPFN v2 (>=2.0) 강제 사용: v0.1.9가 감지되면 USE_TABPFN=False
  - 크로스 섹션 5분위 레이블 (날짜별 10종목 상대 순위)
"""
import os
import sys
import platform
import warnings
import time
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from dotenv import load_dotenv

import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import accuracy_score, log_loss

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

warnings.filterwarnings("ignore")

# ─── 한글 폰트 설정 ───────────────────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    try:
        import koreanize_matplotlib
    except ImportError:
        pass
plt.rcParams["axes.unicode_minus"] = False

# ─── 경로 설정 ────────────────────────────────────────────────────────────────
def _find_base_dir() -> Path:
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / "data" / "panels").is_dir():
            return candidate
    return Path("C:/Users/gorhk/최종 프로젝트/finance_project/김재천/black_litterman")

BASE_DIR     = _find_base_dir()
NOTEBOOK_DIR = BASE_DIR / "Test"
DATA_DIR     = BASE_DIR / "data"
OUT_DIR      = NOTEBOOK_DIR / "output_panel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── .env에서 TabPFN 토큰 로드 → os.environ에 명시적 설정 ────────────────────
load_dotenv(dotenv_path=NOTEBOOK_DIR / ".env")

token = os.getenv("TABPFN_TOKEN", "")
if not token or token == "여기에_토큰_입력":
    print("[경고] TABPFN_TOKEN이 설정되지 않았습니다.")
    print(f"  {NOTEBOOK_DIR / '.env'} 파일을 열어 토큰을 입력하세요.")
    print("  토큰 발급: https://app.priorlabs.ai")
    USE_TABPFN = False
else:
    # v2 API는 fit() 호출 전에 TABPFN_TOKEN 환경변수가 반드시 설정돼야 함
    os.environ["TABPFN_TOKEN"] = token
    print(f"[OK] TABPFN_TOKEN 로드 및 환경변수 설정 완료 ({token[:6]}...)")
    USE_TABPFN = True

# ─── TabPFN v2 강제 임포트 ────────────────────────────────────────────────────
# v0.1.9가 감지되면 USE_TABPFN=False (v2 강제)
# venv 커널 사용 시 v7.1.1(v2 계열) 로드됨
_TabPFNCls = None
if USE_TABPFN:
    try:
        import tabpfn as _tabpfn_pkg
        from tabpfn import TabPFNClassifier as _TabPFNCls

        _ver = getattr(_tabpfn_pkg, "__version__", "0.0.0")
        _major = int(_ver.split(".")[0])

        if _major >= 2:
            print(f"[OK] TabPFN v{_ver} (v2 API) 임포트 성공")
        else:
            # v0.1.9 등 구버전 → v2 강제이므로 비활성화
            print(f"[경고] TabPFN v{_ver} (구버전) 감지 — v2를 강제하므로 TabPFN 비활성화")
            print("  올바른 커널(venv)을 선택했는지 확인하세요.")
            print("  Jupyter: Kernel → Change Kernel → finance-project (.venv)")
            USE_TABPFN = False
            _TabPFNCls = None
    except Exception as e:
        print(f"[경고] TabPFN 임포트 실패: {e}")
        USE_TABPFN = False
        _TabPFNCls = None

# ─── 파라미터 ─────────────────────────────────────────────────────────────────
IT_TICKERS    = ["MSFT", "INTC", "ORCL", "AAPL", "CSCO",
                 "IBM", "QCOM", "TXN", "CRM", "ADBE"]
FEATURE_COLS  = [
    "log_return_1d", "simple_return_1d",
    "mom_1m", "mom_3m", "mom_6m", "mom_12m", "mom_12m_skip_1m",
    "vol_20d_ann", "vol_60d_ann", "vol_252d_ann",
    "mkt_rf", "smb", "hml", "rmw", "cma", "rf", "mom_factor",
]
TARGET_COL    = "fwd_ret_21d"
DATA_START    = pd.Timestamp("2020-12-01")
DATA_END      = pd.Timestamp("2025-12-31")
IS_DAYS       = 252
EMBARGO_DAYS  = 21
OOS_DAYS      = 21
STEP_SIZE     = 21
N_CLASSES     = 5
N_OPTUNA_TRIALS = 30
RANDOM_STATE  = 42


# ─── 함수 정의 ────────────────────────────────────────────────────────────────

def load_ticker_csv(ticker: str) -> pd.DataFrame:
    """panels/{ticker}.csv 로드 → fwd_ret_21d 계산."""
    path = DATA_DIR / "panels" / f"{ticker}.csv"
    df = pd.read_csv(path, index_col="date", parse_dates=True)
    df = df[(df.index >= DATA_START) & (df.index <= DATA_END)].copy()
    df.sort_index(inplace=True)
    df[TARGET_COL] = df["adj_close"].shift(-OOS_DAYS) / df["adj_close"] - 1
    return df


def make_wf_windows(
    dates: pd.DatetimeIndex,
    is_days: int = IS_DAYS,
    embargo: int = EMBARGO_DAYS,
    oos_days: int = OOS_DAYS,
    step: int = STEP_SIZE,
) -> List[Tuple[pd.DatetimeIndex, pd.DatetimeIndex]]:
    """Walk-forward 윈도우 생성."""
    windows, n, i = [], len(dates), 0
    while True:
        is_end    = i + is_days
        oos_start = is_end + embargo
        oos_end   = oos_start + oos_days
        if oos_end > n:
            break
        purge = is_end - oos_days
        is_w  = dates[i:purge]
        oos_w = dates[oos_start:oos_end]
        if len(is_w) > 0 and len(oos_w) > 0:
            windows.append((is_w, oos_w))
        i += step
    return windows


def build_panel(
    ticker_data: Dict[str, pd.DataFrame],
    dates: pd.DatetimeIndex,
    require_target: bool = True,
) -> pd.DataFrame:
    """
    여러 종목 데이터를 세로로 스택하여 패널 DataFrame 반환.
    ticker 컬럼 포함. require_target=True이면 fwd_ret_21d NaN 행 제거.
    """
    frames = []
    for ticker, df in ticker_data.items():
        sub = df.loc[df.index.isin(dates)].copy()
        if require_target:
            sub = sub.dropna(subset=FEATURE_COLS + [TARGET_COL])
        sub["ticker"] = ticker
        frames.append(sub)
    return pd.concat(frames).sort_index()


def make_cs_quintile_labels(
    df_is: pd.DataFrame,
) -> Tuple[np.ndarray, Dict[int, float]]:
    """
    크로스 섹션 5분위 레이블 생성.

    같은 날짜에 10종목을 fwd_ret_21d 기준으로 순위를 매겨 5분위 부여.
    예: 2022-01-03에 10종목 중 수익률 상위 20% → 분위 4

    기존 per-ticker 방식과의 차이:
      기존: MSFT의 과거 252일치 수익률 중 어느 분위인가?
      패널: 오늘 날짜에 10종목 중 어느 상대 순위인가?
    """
    def _quintile_per_date(group):
        if len(group) < N_CLASSES:
            return pd.Series(0, index=group.index)
        return pd.qcut(
            group[TARGET_COL].rank(method="first"),
            q=N_CLASSES, labels=False
        ).astype(int)

    labels_s = (
        df_is.groupby(df_is.index, group_keys=False)
             .apply(_quintile_per_date)
    )
    labels_s = labels_s.reindex(df_is.index).fillna(0).astype(int)
    r_bar = {
        k: float(df_is.loc[labels_s == k, TARGET_COL].mean())
        for k in range(N_CLASSES)
    }
    return labels_s.values, r_bar


def compute_Q_Omega(
    proba: np.ndarray,
    r_bar: Dict[int, float],
    n_classes: int = N_CLASSES,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    B-L view Q(기대수익률)와 Ω(불확실성) 계산.
    Q_i = Σ_k proba[i,k] * r_bar[k]
    Ω_i = Σ_k proba[i,k] * (r_bar[k] - Q_i)²
    """
    r_arr = np.array([r_bar.get(k, 0.0) for k in range(n_classes)])
    Q     = proba @ r_arr
    Omega = np.sum(proba * (r_arr[np.newaxis, :] - Q[:, np.newaxis]) ** 2, axis=1)
    return Q, Omega


def _xgb_objective(trial, X_tr, y_tr, X_val, y_val):
    """Optuna XGBoost 목적 함수."""
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators",   100, 500),
        "max_depth"         : trial.suggest_int("max_depth",       3,   7),
        "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"         : trial.suggest_float("subsample",     0.5, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight"  : trial.suggest_int("min_child_weight", 1, 10),
        "gamma"             : trial.suggest_float("gamma",         0.0, 3.0),
        "reg_lambda"        : trial.suggest_float("reg_lambda",    0.1, 10.0, log=True),
        "reg_alpha"         : trial.suggest_float("reg_alpha",     1e-5, 1.0, log=True),
        "objective"         : "multi:softprob",
        "num_class"         : N_CLASSES,
        "eval_metric"       : "mlogloss",
        "tree_method"       : "hist",
        "random_state"      : RANDOM_STATE,
        "n_jobs"            : -1,
        "verbosity"         : 0,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    proba_val = model.predict_proba(X_val)
    return log_loss(y_val, proba_val, labels=list(range(N_CLASSES)))


def train_xgb(
    X_is: np.ndarray,
    y_is: np.ndarray,
    X_oos: np.ndarray,
    window_id: int = 0,
    n_trials: int = N_OPTUNA_TRIALS,
) -> Tuple[np.ndarray, dict]:
    """패널 IS 데이터로 XGBoost Optuna 학습 → 패널 OOS 확률 반환."""
    n_is  = len(X_is)
    split = int(n_is * 0.8)
    X_tr, X_val = X_is[:split], X_is[split:]
    y_tr, y_val = y_is[:split], y_is[split:]

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE + window_id)
    study   = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(
        lambda trial: _xgb_objective(trial, X_tr, y_tr, X_val, y_val),
        n_trials=n_trials,
        show_progress_bar=False,
    )

    best_params = study.best_params
    best_params.update({
        "objective"    : "multi:softprob",
        "num_class"    : N_CLASSES,
        "eval_metric"  : "mlogloss",
        "tree_method"  : "hist",
        "random_state" : RANDOM_STATE,
        "n_jobs"       : -1,
        "verbosity"    : 0,
    })
    final_model = xgb.XGBClassifier(**best_params)
    final_model.fit(X_is, y_is, verbose=False)
    return final_model.predict_proba(X_oos), best_params


def train_tabpfn_v2(
    X_is: np.ndarray,
    y_is: np.ndarray,
    X_oos: np.ndarray,
) -> np.ndarray:
    """
    TabPFN v2 학습 → OOS 확률 반환.
    TABPFN_TOKEN 환경변수를 사용해 인증 (로컬 모델 다운로드 또는 API 호출).
    행 제한 없음 — 패널 IS ~2,310행에서 사용 가능.
    """
    clf   = _TabPFNCls()
    clf.fit(X_is, y_is)
    proba = clf.predict_proba(X_oos)

    # 클래스 순서 0~4 보장
    classes = clf.classes_
    if not np.array_equal(classes, np.arange(N_CLASSES)):
        order = np.argsort(classes)
        proba = proba[:, order]
    return proba


def compute_eval_metrics(
    Q_pred: np.ndarray,
    y_pred_cls: np.ndarray,
    y_true_cls: np.ndarray,
    y_true_ret: np.ndarray,
) -> Dict[str, float]:
    """Accuracy, Hit Rate, IC 계산."""
    valid = ~np.isnan(y_true_ret)
    if valid.sum() == 0:
        return {"accuracy": np.nan, "hit_rate": np.nan, "ic": np.nan}
    acc      = accuracy_score(y_true_cls[valid], y_pred_cls[valid])
    hit_rate = np.mean(np.sign(Q_pred[valid]) == np.sign(y_true_ret[valid]))
    ic, _    = spearmanr(Q_pred[valid], y_true_ret[valid])
    return {"accuracy": acc, "hit_rate": hit_rate, "ic": ic if not np.isnan(ic) else 0.0}


print("Section 0 PASS [OK] — 함수 정의 완료 (패널 버전)")
print(f"  USE_TABPFN = {USE_TABPFN}")
print(f"  IT_TICKERS : {IT_TICKERS}")
print(f"  IS_DAYS={IS_DAYS}, EMBARGO={EMBARGO_DAYS}, OOS={OOS_DAYS}")
print(f"  출력 폴더: {OUT_DIR}")


[OK] TABPFN_TOKEN 로드 및 환경변수 설정 완료 (eyJhbG...)
[OK] TabPFN v7.1.1 (v2 API) 임포트 성공
Section 0 PASS [OK] — 함수 정의 완료 (패널 버전)
  USE_TABPFN = True
  IT_TICKERS : ['MSFT', 'INTC', 'ORCL', 'AAPL', 'CSCO', 'IBM', 'QCOM', 'TXN', 'CRM', 'ADBE']
  IS_DAYS=252, EMBARGO=21, OOS=21
  출력 폴더: C:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\Test\output_panel


## Section 1: 데이터 로드 및 설정 확인

In [2]:
"""
Section 1: 데이터 로드 및 설정 확인 (패널 버전)
  - 10종목 CSV 로드 및 피처 검증
  - 첫 번째 윈도우로 패널 크기 미리 확인
  - ticker_data, windows 변수는 이후 Section 2에서 재사용
"""
print("=" * 60)
print("SECTION 1: 데이터 로드 및 설정 확인 (패널 버전)")
print("=" * 60)

# ── 전체 종목 데이터 로드 (이후 셀에서 재사용) ────────────────────────────
ticker_data = {}
for ticker in IT_TICKERS:
    df = load_ticker_csv(ticker)

    missing_feats = [c for c in FEATURE_COLS if c not in df.columns]
    assert not missing_feats, f"{ticker} 피처 누락: {missing_feats}"
    assert TARGET_COL in df.columns, f"{ticker} target 컬럼 없음"

    ticker_data[ticker] = df
    n_total  = len(df)
    n_valid  = df[TARGET_COL].notna().sum()
    date_rng = f"{df.index[0].date()} ~ {df.index[-1].date()}"
    print(f"  {ticker:5s} : {n_total}행, target 유효 {n_valid}행, 기간 {date_rng}")

# ── Walk-forward 윈도우 생성 (이후 셀에서 재사용) ─────────────────────────
all_dates = list(ticker_data.values())[0].index
windows   = make_wf_windows(all_dates)
print(f"\n  총 walk-forward 윈도우 : {len(windows)}개")
print(f"  첫 번째 IS  : {windows[0][0][0].date()} ~ {windows[0][0][-1].date()}")
print(f"  첫 번째 OOS : {windows[0][1][0].date()} ~ {windows[0][1][-1].date()}")
print(f"  마지막  OOS : {windows[-1][1][0].date()} ~ {windows[-1][1][-1].date()}")

# ── 첫 번째 윈도우로 패널 크기 미리 확인 ─────────────────────────────────
is_w0, oos_w0  = windows[0]
df_is_sample   = build_panel(ticker_data, is_w0,  require_target=True)
df_oos_sample  = build_panel(ticker_data, oos_w0, require_target=False)

print(f"\n[패널 크기 확인 — 첫 번째 윈도우]")
print(f"  IS 패널  : {len(df_is_sample)}행  ({len(is_w0)}일 × {len(IT_TICKERS)}종목)")
print(f"  OOS 패널 : {len(df_oos_sample)}행")

# ── 크로스 섹션 5분위 레이블 샘플 확인 ────────────────────────────────────
y_is_cls_sample, r_bar_sample = make_cs_quintile_labels(df_is_sample)
unique, counts = np.unique(y_is_cls_sample, return_counts=True)
print(f"\n[크로스 섹션 5분위 레이블 분포 — IS]")
for cls, cnt in zip(unique, counts):
    print(f"  분위 {cls}: {cnt}행, r_bar = {r_bar_sample[cls]:.5f}")

print(f"\n[기존 per-ticker 대비 IS 크기 비교]")
print(f"  기존 방식 : 252 × 1종목 = 252행")
print(f"  패널 방식 : {len(df_is_sample)}행 ({len(IT_TICKERS)}종목 스택)")

print("\nSECTION 1 PASS [OK]")


SECTION 1: 데이터 로드 및 설정 확인 (패널 버전)
  MSFT  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  INTC  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  ORCL  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  AAPL  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  CSCO  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  IBM   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  QCOM  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  TXN   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  CRM   : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30
  ADBE  : 1276행, target 유효 1255행, 기간 2020-12-01 ~ 2025-12-30

  총 walk-forward 윈도우 : 47개
  첫 번째 IS  : 2020-12-01 ~ 2021-10-29
  첫 번째 OOS : 2021-12-31 ~ 2022-01-31
  마지막  OOS : 2025-11-06 ~ 2025-12-05

[패널 크기 확인 — 첫 번째 윈도우]
  IS 패널  : 2310행  (231일 × 10종목)
  OOS 패널 : 210행

[크로스 섹션 5분위 레이블 분포 — IS]
  분위 0: 462행, r_bar = -0.05178
  분위 1: 462행, r_bar = -0.00913
  분위 2: 462행, r_bar = 0.02098
  분위 3: 462행, r_bar = 0.0526

In [6]:
# IS 패널 원본 (피처 + target + ticker)
display(df_is_sample[FEATURE_COLS + [TARGET_COL, "ticker"]].head(20))

# 레이블이 붙은 형태로 보기
df_labeled = df_is_sample[FEATURE_COLS + [TARGET_COL, "ticker"]].copy()
df_labeled["quintile"] = y_is_cls_sample
display(df_labeled.head(30))

# 날짜별로 10종목이 어떻게 분위 배분되는지 확인
print("\n[날짜별 분위 분포 샘플]")
display(
    df_labeled.groupby([df_labeled.index, "quintile"])
              .size()
              .unstack(fill_value=0)
              .head(5)
)

,log_return_1d,simple_return_1d,mom_1m,mom_3m,mom_6m,mom_12m,mom_12m_skip_1m,vol_20d_ann,vol_60d_ann,vol_252d_ann,mkt_rf,smb,hml,rmw,cma,rf,mom_factor,fwd_ret_21d,ticker
date,,,,,,,,,,,,,,,,,,,
2020-12-01,0.009947,0.009997,0.070658,-0.046174,0.172314,0.461053,0.364631,0.300911,0.324721,0.439140,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.028722,MSFT
2020-12-01,-0.018270,-0.018104,0.039092,-0.141867,0.379222,0.499068,0.442671,0.425784,0.372960,0.513010,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,-0.077978,CRM
2020-12-01,0.030362,0.030827,0.129264,-0.083830,0.515174,0.874316,0.659767,0.293882,0.411060,0.463603,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.081242,AAPL
2020-12-01,0.001420,0.001421,0.071684,-0.092433,0.228842,0.582659,0.476796,0.411534,0.385652,0.478493,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.043765,ADBE
2020-12-01,0.017517,0.017671,0.046872,0.022855,0.107257,0.078359,0.030077,0.157541,0.210316,0.400959,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.101294,ORCL
2020-12-01,0.024718,0.025026,0.127382,-0.017120,-0.188446,-0.119106,-0.218638,0.223140,0.329336,0.532286,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.005246,INTC
2020-12-01,0.027412,0.027791,0.226167,0.246375,0.807991,0.887980,0.539741,0.522581,0.417882,0.520373,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.011487,QCOM
2020-12-01,0.013796,0.013891,0.130714,0.135931,0.303341,0.416792,0.253006,0.261303,0.272709,0.430646,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.003915,TXN
2020-12-01,0.012015,0.012087,0.212813,0.045482,-0.056504,0.007524,-0.169267,0.302450,0.259939,0.422779,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.027791,CSCO


,log_return_1d,simple_return_1d,mom_1m,mom_3m,mom_6m,mom_12m,mom_12m_skip_1m,vol_20d_ann,vol_60d_ann,vol_252d_ann,mkt_rf,smb,hml,rmw,cma,rf,mom_factor,fwd_ret_21d,ticker,quintile
date,,,,,,,,,,,,,,,,,,,,
2020-12-01,0.009947,0.009997,0.070658,-0.046174,0.172314,0.461053,0.364631,0.300911,0.324721,0.439140,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.028722,MSFT,3
2020-12-01,-0.018270,-0.018104,0.039092,-0.141867,0.379222,0.499068,0.442671,0.425784,0.372960,0.513010,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,-0.077978,CRM,0
2020-12-01,0.030362,0.030827,0.129264,-0.083830,0.515174,0.874316,0.659767,0.293882,0.411060,0.463603,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.081242,AAPL,4
2020-12-01,0.001420,0.001421,0.071684,-0.092433,0.228842,0.582659,0.476796,0.411534,0.385652,0.478493,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.043765,ADBE,3
2020-12-01,0.017517,0.017671,0.046872,0.022855,0.107257,0.078359,0.030077,0.157541,0.210316,0.400959,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.101294,ORCL,4
2020-12-01,0.024718,0.025026,0.127382,-0.017120,-0.188446,-0.119106,-0.218638,0.223140,0.329336,0.532286,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.005246,INTC,1
2020-12-01,0.027412,0.027791,0.226167,0.246375,0.807991,0.887980,0.539741,0.522581,0.417882,0.520373,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.011487,QCOM,1
2020-12-01,0.013796,0.013891,0.130714,0.135931,0.303341,0.416792,0.253006,0.261303,0.272709,0.430646,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.003915,TXN,0
2020-12-01,0.012015,0.012087,0.212813,0.045482,-0.056504,0.007524,-0.169267,0.302450,0.259939,0.422779,0.0099,-0.0006,0.0052,0.0104,0.0053,0.0,-0.0078,0.027791,CSCO,2



[날짜별 분위 분포 샘플]


quintile,0,1,2,3,4
date,,,,,
2020-12-01,2,2,2,2,2
2020-12-02,2,2,2,2,2
2020-12-03,2,2,2,2,2
2020-12-04,2,2,2,2,2
2020-12-07,2,2,2,2,2


In [7]:
X_is = df_is_sample[FEATURE_COLS].values.astype(float)
print(f"X_is shape: {X_is.shape}")   # (2310, 17)
print(pd.DataFrame(X_is, columns=FEATURE_COLS).describe())

X_is shape: (2310, 17)
       log_return_1d  simple_return_1d       mom_1m       mom_3m       mom_6m  \
count    2310.000000       2310.000000  2310.000000  2310.000000  2310.000000   
mean        0.000940          0.001070     0.024149     0.074704     0.157499   
std         0.016102          0.016025     0.066857     0.108804     0.165713   
min        -0.124187         -0.116786    -0.220724    -0.202067    -0.269938   
25%        -0.007082         -0.007057    -0.015270     0.003145     0.059822   
50%         0.000897          0.000898     0.024925     0.073605     0.156942   
75%         0.009897          0.009946     0.064398     0.145185     0.252429   
max         0.069633          0.072115     0.316056     0.445768     0.874942   

           mom_12m  mom_12m_skip_1m  vol_20d_ann  vol_60d_ann  vol_252d_ann  \
count  2310.000000      2310.000000  2310.000000  2310.000000   2310.000000   
mean      0.363743         0.336337     0.239811     0.257144      0.353445   
std       

## Section 2: Walk-Forward 실행 (XGBoost + TabPFN)

In [3]:
"""
Section 2: Walk-Forward 메인 루프 (패널 버전)

핵심 변경 사항 (기존 per-ticker 대비):
  ─ 기존: for window: for ticker: → 10개 모델 / window
  ─ 패널: for window: → 1개 모델 / window  (약 10배 빠름)

각 window 처리 흐름:
  1. build_panel()           → 10종목 IS 데이터를 세로 스택 (~2,520행)
  2. make_cs_quintile_labels → 날짜별 상대 순위 5분위
  3. train_xgb()             → XGBoost 패널 전체 1회 학습
  4. train_tabpfn_v2()       → TabPFN v2 패널 전체 1회 학습 (토큰 있을 때만)
  5. OOS 예측 결과를 종목별로 분리하여 records 저장
"""
print("=" * 60)
print("SECTION 2: Walk-Forward 실행 (패널 버전)")
print(f"  XGBoost : 항상 실행")
if USE_TABPFN:
    print(f"  TabPFN  : 실행 (v2 API)")
else:
    # USE_TABPFN=False 원인: 토큰 미설정 또는 v2 아닌 버전 감지
    # → Section 0 출력에서 정확한 원인 확인 가능
    _reason = getattr(globals().get('_tabpfn_pkg', None), '__version__', '?')
    print(f"  TabPFN  : 비활성화 (Section 0 출력 확인, 감지된 버전={_reason})")
    print(f"            venv 커널 필요: Kernel → Change Kernel → finance-project (.venv)")
print("=" * 60)

# ticker_data, windows → Section 1에서 정의됨
records = []
t0      = time.time()

for w_idx, (is_dates, oos_dates) in enumerate(windows):
    is_start  = is_dates[0].date()
    oos_start = oos_dates[0].date()
    print(f"\n[Window {w_idx+1:3d}/{len(windows)}] "
          f"IS {is_start} ~ {is_dates[-1].date()} | "
          f"OOS {oos_start} ~ {oos_dates[-1].date()}")

    # ── 1. 패널 데이터 구성 ───────────────────────────────────────────────
    df_is  = build_panel(ticker_data, is_dates,  require_target=True)
    df_oos = build_panel(ticker_data, oos_dates, require_target=False)

    if len(df_is) < N_CLASSES * len(IT_TICKERS):
        print(f"  IS 데이터 부족 ({len(df_is)}행), 스킵")
        continue
    if len(df_oos) == 0:
        print(f"  OOS 데이터 없음, 스킵")
        continue

    X_is  = df_is[FEATURE_COLS].values.astype(float)
    X_oos = df_oos[FEATURE_COLS].fillna(0).values.astype(float)

    # ── 2. IS 크로스 섹션 5분위 레이블 생성 ──────────────────────────────
    try:
        y_is_cls, r_bar = make_cs_quintile_labels(df_is)
    except Exception as e:
        print(f"  IS 분위 생성 실패: {e}, 스킵")
        continue

    # ── 3. OOS 크로스 섹션 5분위 레이블 (logloss 계산용) ─────────────────
    # 날짜별로 실제 수익률 기준 상대 순위 5분위 할당
    y_oos_ret = df_oos[TARGET_COL].values
    y_oos_cls = np.zeros(len(df_oos), dtype=int)

    df_oos_pos         = df_oos.copy()
    df_oos_pos["_pos"] = np.arange(len(df_oos))

    for _, grp in df_oos_pos.groupby(df_oos_pos.index):
        valid_mask = grp[TARGET_COL].notna()
        if valid_mask.sum() < N_CLASSES:
            continue
        try:
            cls_grp = pd.qcut(
                grp.loc[valid_mask, TARGET_COL].rank(method="first"),
                q=N_CLASSES, labels=False,
            ).astype(int)
            y_oos_cls[grp.loc[valid_mask, "_pos"].values] = cls_grp.values
        except Exception:
            pass  # 분위 생성 실패 → 0 유지

    valid_oos = ~np.isnan(y_oos_ret)   # OOS logloss 계산 마스크

    # ── 4. XGBoost (패널 전체 1개 학습) ──────────────────────────────────
    try:
        proba_xgb, _ = train_xgb(X_is, y_is_cls, X_oos, window_id=w_idx)
        Q_xgb, Om_xgb   = compute_Q_Omega(proba_xgb, r_bar)
        pred_cls_xgb     = np.argmax(proba_xgb, axis=1)
        ll_xgb = (
            log_loss(y_oos_cls[valid_oos], proba_xgb[valid_oos],
                     labels=list(range(N_CLASSES)))
            if valid_oos.sum() > 0 else np.nan
        )
        print(f"  XGB   : IS={len(df_is)}행  OOS={len(df_oos)}행  LogLoss={ll_xgb:.4f}")
    except Exception as e:
        print(f"  XGB 오류: {e}")
        proba_xgb        = np.full((len(df_oos), N_CLASSES), 1.0 / N_CLASSES)
        Q_xgb, Om_xgb    = np.zeros(len(df_oos)), np.ones(len(df_oos)) * 1e-4
        pred_cls_xgb     = np.zeros(len(df_oos), dtype=int)
        ll_xgb           = np.nan

    # ── 5. TabPFN v2 (토큰 있을 때만, 패널 전체 1개 학습) ────────────────
    ll_tab = np.nan
    if USE_TABPFN:
        try:
            proba_tab = train_tabpfn_v2(X_is, y_is_cls, X_oos)
            Q_tab, Om_tab   = compute_Q_Omega(proba_tab, r_bar)
            pred_cls_tab     = np.argmax(proba_tab, axis=1)
            ll_tab = (
                log_loss(y_oos_cls[valid_oos], proba_tab[valid_oos],
                         labels=list(range(N_CLASSES)))
                if valid_oos.sum() > 0 else np.nan
            )
            print(f"  TabPFN: LogLoss={ll_tab:.4f}")
        except Exception as e:
            print(f"  TabPFN 오류: {e}")
            proba_tab        = np.full((len(df_oos), N_CLASSES), 1.0 / N_CLASSES)
            Q_tab, Om_tab    = np.zeros(len(df_oos)), np.ones(len(df_oos)) * 1e-4
            pred_cls_tab     = np.zeros(len(df_oos), dtype=int)
            ll_tab           = np.nan

    # ── 6. OOS 결과 기록 (종목별 분리) ───────────────────────────────────
    for j, (idx, row) in enumerate(df_oos.iterrows()):
        base = {
            "window_id"  : w_idx,
            "date"       : idx,
            "ticker"     : row["ticker"],
            "actual_ret" : float(y_oos_ret[j]),
            "true_cls"   : int(y_oos_cls[j]),
        }
        records.append({
            **base, "model"    : "xgb",
            "Q"      : float(Q_xgb[j]),
            "Omega"  : float(Om_xgb[j]),
            "pred_cls": int(pred_cls_xgb[j]),
            "logloss" : ll_xgb,
        })
        if USE_TABPFN:
            records.append({
                **base, "model"    : "tabpfn",
                "Q"      : float(Q_tab[j]),
                "Omega"  : float(Om_tab[j]),
                "pred_cls": int(pred_cls_tab[j]),
                "logloss" : ll_tab,
            })

elapsed_total = time.time() - t0
print(f"\n총 소요 시간 : {elapsed_total / 60:.1f}분")
print(f"총 기록 수   : {len(records)}개")
print("SECTION 2 PASS [OK]")


SECTION 2: Walk-Forward 실행 (패널 버전)
  XGBoost : 항상 실행
  TabPFN  : 실행 (v2 API)

[Window   1/47] IS 2020-12-01 ~ 2021-10-29 | OOS 2021-12-31 ~ 2022-01-31
  XGB   : IS=2310행  OOS=210행  LogLoss=1.6939


tabpfn-v2.6-classifier-v2.6_default.ckpt:   0%|          | 0.00/43.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

  TabPFN: LogLoss=1.6907

[Window   2/47] IS 2020-12-31 ~ 2021-11-30 | OOS 2022-02-01 ~ 2022-03-02
  XGB   : IS=2310행  OOS=210행  LogLoss=1.5632
  TabPFN: LogLoss=1.6150

[Window   3/47] IS 2021-02-02 ~ 2021-12-30 | OOS 2022-03-03 ~ 2022-03-31
  XGB   : IS=2310행  OOS=210행  LogLoss=1.5734
  TabPFN: LogLoss=1.6270

[Window   4/47] IS 2021-03-04 ~ 2022-01-31 | OOS 2022-04-01 ~ 2022-05-02
  XGB   : IS=2310행  OOS=210행  LogLoss=1.6276
  TabPFN: LogLoss=1.6774

[Window   5/47] IS 2021-04-05 ~ 2022-03-02 | OOS 2022-05-03 ~ 2022-06-01
  XGB   : IS=2310행  OOS=210행  LogLoss=1.6845
  TabPFN: LogLoss=1.7162

[Window   6/47] IS 2021-05-04 ~ 2022-03-31 | OOS 2022-06-02 ~ 2022-07-01
  XGB   : IS=2310행  OOS=210행  LogLoss=1.6825
  TabPFN: LogLoss=1.6568

[Window   7/47] IS 2021-06-03 ~ 2022-05-02 | OOS 2022-07-05 ~ 2022-08-02
  XGB   : IS=2310행  OOS=210행  LogLoss=1.6012
  TabPFN: LogLoss=1.6988

[Window   8/47] IS 2021-07-02 ~ 2022-06-01 | OOS 2022-08-03 ~ 2022-08-31
  XGB   : IS=2310행  OOS=210행  LogLoss

## Section 3: 분류 성능 평가

In [4]:
"""
Section 3: 분류 성능 평가 (패널 버전)
  - Accuracy, Hit Rate, IC, LogLoss
  - 종목 × 모델별 요약표
  - 모델 전체 평균 비교 (XGBoost vs TabPFN v2)
  - USE_TABPFN=False 이면 XGBoost 단독 평가
"""
print("=" * 60)
print("SECTION 3: 분류 성능 평가 (패널 버전)")
print("=" * 60)

df_res   = pd.DataFrame(records)
df_valid = df_res.dropna(subset=["actual_ret"])

# ── 종목 × 모델별 집계 ────────────────────────────────────────────────────
eval_rows = []
for (ticker, model), grp in df_valid.groupby(["ticker", "model"]):
    q_pred   = grp["Q"].values
    cls_pred = grp["pred_cls"].values
    cls_true = grp["true_cls"].values
    ret_true = grp["actual_ret"].values

    metrics = compute_eval_metrics(q_pred, cls_pred, cls_true, ret_true)
    eval_rows.append({
        "ticker"   : ticker,
        "model"    : model,
        "accuracy" : metrics["accuracy"],
        "hit_rate" : metrics["hit_rate"],
        "ic"       : metrics["ic"],
        "logloss"  : grp["logloss"].mean(),
        "n_oos"    : len(grp),
    })

df_eval = pd.DataFrame(eval_rows)

# ── 종목 × 모델 성능표 ────────────────────────────────────────────────────
print("\n[종목 × 모델 성능표]")
print(df_eval.to_string(index=False, float_format="{:.4f}".format))

# ── 모델 전체 평균 비교 ───────────────────────────────────────────────────
print("\n[모델 전체 평균 비교]")
agg = df_eval.groupby("model")[["accuracy", "hit_rate", "ic", "logloss"]].mean()
agg["accuracy_vs_random"] = agg["accuracy"] - 0.20
agg["hit_rate_vs_random"] = agg["hit_rate"] - 0.50
print(agg.to_string(float_format="{:.4f}".format))
print("  (accuracy 기준선=0.20, hit_rate 기준선=0.50, IC 기준선=0.0)")

if not USE_TABPFN:
    print("\n  [참고] TabPFN 토큰 미설정 → XGBoost 단독 결과입니다.")
    print("  Test/.env에 TABPFN_TOKEN 입력 후 재실행하면 TabPFN v2 결과 추가됩니다.")

print("\nSECTION 3 PASS [OK]")


SECTION 3: 분류 성능 평가 (패널 버전)

[종목 × 모델 성능표]
ticker  model  accuracy  hit_rate      ic  logloss  n_oos
  AAPL tabpfn    0.1701    0.5703 -0.0824   1.6963    982
  AAPL    xgb    0.2077    0.5102 -0.0725   1.6553    982
  ADBE tabpfn    0.1802    0.4766 -0.1033   1.6963    982
  ADBE    xgb    0.1955    0.4430 -0.1267   1.6553    982
   CRM tabpfn    0.2363    0.5285  0.0041   1.6963    982
   CRM    xgb    0.2495    0.4878 -0.0084   1.6553    982
  CSCO tabpfn    0.1568    0.5224 -0.0769   1.6963    982
  CSCO    xgb    0.2006    0.5163 -0.2243   1.6553    982
   IBM tabpfn    0.1813    0.5448 -0.0683   1.6963    982
   IBM    xgb    0.2251    0.6161  0.0383   1.6553    982
  INTC tabpfn    0.2312    0.5204  0.0080   1.6963    982
  INTC    xgb    0.1965    0.5193 -0.0342   1.6553    982
  MSFT tabpfn    0.2016    0.5448  0.0253   1.6963    982
  MSFT    xgb    0.2312    0.5020 -0.1036   1.6553    982
  ORCL tabpfn    0.2261    0.3849 -0.1101   1.6963    982
  ORCL    xgb    0.2006    0.

## Section 4: 결과 저장 (Step4 입력 파일)

In [5]:
"""
Section 4: Q, Ω CSV 저장 (Step4 B-L 입력용)

저장 경로: Test/output_panel/   (기존 Step3의 output/ 와 별도)

저장 파일:
  Q_xgb.csv          — date × ticker, 연환산 21일 예측 수익률 (XGB)
  Omega_xgb.csv      — date × ticker, 연환산 예측 불확실성   (XGB)
  Q_tabpfn.csv       — USE_TABPFN=True 때만 저장
  Omega_tabpfn.csv   — USE_TABPFN=True 때만 저장
  classification_stats.csv  — 종목 × 모델 성능 요약
"""
print("=" * 60)
print("SECTION 4: 결과 저장")
print("=" * 60)

ANN_FACTOR  = 252 / OOS_DAYS    # 연환산 수익률 배수  (= 12)
ANN_FACTOR2 = ANN_FACTOR ** 2   # 연환산 분산 배수    (= 144)

# 저장할 모델 목록 (TabPFN은 토큰 있을 때만)
models_to_save = ["xgb"] + (["tabpfn"] if USE_TABPFN else [])

for model_name in models_to_save:
    df_m = df_res[df_res["model"] == model_name][
        ["date", "ticker", "Q", "Omega"]
    ].copy()

    # 연환산 변환 (OOS 21일 → 연율)
    df_m["Q"]     *= ANN_FACTOR
    df_m["Omega"] *= ANN_FACTOR2

    # date × ticker 피벗
    q_pivot     = df_m.pivot_table(index="date", columns="ticker", values="Q")
    omega_pivot = df_m.pivot_table(index="date", columns="ticker", values="Omega")

    # 컬럼 순서 통일 (IT_TICKERS 순서)
    q_pivot     = q_pivot.reindex(columns=IT_TICKERS)
    omega_pivot = omega_pivot.reindex(columns=IT_TICKERS)

    q_path     = OUT_DIR / f"Q_{model_name}.csv"
    omega_path = OUT_DIR / f"Omega_{model_name}.csv"
    q_pivot.to_csv(q_path,     encoding="utf-8-sig")
    omega_pivot.to_csv(omega_path, encoding="utf-8-sig")

    q_vals = q_pivot.values
    valid_vals = q_vals[~np.isnan(q_vals)]
    print(f"  [{model_name}] Q shape={q_pivot.shape}"
          f"  범위=[{valid_vals.min():.4f}, {valid_vals.max():.4f}]")
    print(f"    저장: {q_path.name}, {omega_path.name}")

# 분류 성능 저장
stats_path = OUT_DIR / "classification_stats.csv"
df_eval.to_csv(stats_path, index=False, encoding="utf-8-sig")
print(f"\n  classification_stats.csv 저장 완료 ({len(df_eval)}행)")
print(f"  출력 폴더: {OUT_DIR}")

print("\nSECTION 4 PASS [OK]")
print("\n" + "=" * 60)
print("Step3 (패널) 전체 완료")
print("다음 단계: Step4_IT_BlackLitterman.ipynb 실행")
print("  입력 파일: output_panel/ 폴더의 Q_xgb.csv, Omega_xgb.csv")
print("=" * 60)


SECTION 4: 결과 저장
  [xgb] Q shape=(987, 10)  범위=[-0.8936, 1.1647]
    저장: Q_xgb.csv, Omega_xgb.csv
  [tabpfn] Q shape=(987, 10)  범위=[-1.0507, 1.1595]
    저장: Q_tabpfn.csv, Omega_tabpfn.csv

  classification_stats.csv 저장 완료 (20행)
  출력 폴더: C:\Users\gorhk\최종 프로젝트\finance_project\김재천\black_litterman\Test\output_panel

SECTION 4 PASS [OK]

Step3 (패널) 전체 완료
다음 단계: Step4_IT_BlackLitterman.ipynb 실행
  입력 파일: output_panel/ 폴더의 Q_xgb.csv, Omega_xgb.csv
